El dataset CBIS-DDSM incluye imágenes jpeg, un csv con la información clínica de la lesión, ROI y bounding boxes actualizados (vienen ya las coordenadas de la lesión, si hubiera).

La estructura del dataset es una carpeta con cierto nombre, que es un ID llamado StudyInstanceUID (DICOM). En esa carpeta hay una o varias jpegs.

Los csv pueden contener el patient_id, el image_file_path, y datos como el abnormality_type, pathology, o las coordenadas de la región de interés ROI. También puede traer la ruta relativa del jpeg al que hace referencia. 

Cada paciente tiene varios patient-ID (importante). Debemos tener en cuenta qué fotos son de cada paciente para no usar el mismo paciente en training y en testing.

In [1]:
#Primero habrá que sacar y entender los csv:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


df1 = pd.read_csv('calc_case_description_test_set.csv')
df1

,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00038,2,LEFT,CC,1,calcification,PUNCTATE-PLEOMORPHIC,CLUSTERED,4,BENIGN,2,Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.85935434310203356712688695661986996009/1.3.6.1.4.1.9590.100.1.2.374115997511889073021386151921807063992/000000.dcm,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.161465562211359959230647609981488894942/1.3.6.1.4.1.9590.100.1.2.419081637812053404913157930753972718515/000001.dcm\n,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.161465562211359959230647609981488894942/1.3.6.1.4.1.9590.100.1.2.419081637812053404913157930753972718515/000000.dcm
1,P_00038,2,LEFT,MLO,1,calcification,PUNCTATE-PLEOMORPHIC,CLUSTERED,4,BENIGN,2,Calc-Test_P_00038_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.384159464510350889125645400702639717613/1.3.6.1.4.1.9590.100.1.2.174390361112646747718661211471328897934/000000.dcm,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.29112199613143138535387754440942211739/1.3.6.1.4.1.9590.100.1.2.188613955710170417803011787532523988680/000001.dcm\n,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.29112199613143138535387754440942211739/1.3.6.1.4.1.9590.100.1.2.188613955710170417803011787532523988680/000000.dcm
2,P_00038,2,RIGHT,CC,1,calcification,VASCULAR,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.177706148911820252341905176394069228468/1.3.6.1.4.1.9590.100.1.2.263861248711313923336051913560309963304/000000.dcm,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.200764632211227648028305709420627883105/1.3.6.1.4.1.9590.100.1.2.244876997513875090239564803900035037851/000001.dcm\n,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.200764632211227648028305709420627883105/1.3.6.1.4.1.9590.100.1.2.244876997513875090239564803900035037851/000000.dcm
3,P_00038,2,RIGHT,CC,2,calcification,VASCULAR,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.177706148911820252341905176394069228468/1.3.6.1.4.1.9590.100.1.2.263861248711313923336051913560309963304/000000.dcm,Calc-Test_P_00038_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.248538452013626298441249276382187367143/1.3.6.1.4.1.9590.100.1.2.360550081712464813321995483083632007570/000001.dcm\n,Calc-Test_P_00038_RIGHT_CC_2/1.3.6.1.4.1.9590.100.1.2.248538452013626298441249276382187367143/1.3.6.1.4.1.9590.100.1.2.360550081712464813321995483083632007570/000000.dcm
4,P_00038,2,RIGHT,MLO,1,calcification,VASCULAR,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00038_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.328421320411501709324953698601549885215/1.3.6.1.4.1.9590.100.1.2.44262460211112930513355519060642708846/000000.dcm,Calc-Test_P_00038_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.348569460311013218440657632223354965172/1.3.6.1.4.1.9590.100.1.2.126295284812046209819441424913058621714/000001.dcm\n,Calc-Test_P_00038_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.348569460311013218440657632223354965172/1.3.6.1.4.1.9590.100.1.2.126295284812046209819441424913058621714/000000.dcm
5,P_00038,2,RIGHT,MLO,2,calcification,VASCULAR,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00038_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.328421320411501709324953698601549885215/1.3.6.1.4.1.9590.100.1.2.44262460211112930513355519060642708846/000000.dcm,Calc-Test_P_00038_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.57362695012780927721008884783926968463/1.3.6.1.4.1.9590.100.1.2.326152312412979080929913875463824504551/000001.dcm\n,Calc-Test_P_00038_RIGHT_MLO_2/1.3.6.1.4.1.9590.100.1.2.57362695012780927721008884783926968463/1.3.6.1.4.1.9590.100.1.2.326152312412979080929913875463824504551/000000.dcm
6,P_00041,1,LEFT,CC,2,calcification,LUCENT_CENTER,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00041_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.69692063913639875309068650720468252275/1.3.6.1.4.1.9590.100.1.2.272917492411709393015036949944104292812/000000.dcm,Calc-Test_P_00041

In [2]:
df2 = pd.read_csv('calc_case_description_test_set.csv')
df2.head(8)

,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00016,4,LEFT,CC,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/000000.dcm,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.259596319110047779433501728143778409887/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/000000.dcm,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.259596319110047779433501728143778409887/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/000001.dcm\n
1,P_00016,4,LEFT,MLO,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.125010935311829990124529079264106154392/1.3.6.1.4.1.9590.100.1.2.85952214611170506017891429690540035518/000000.dcm,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.207144238612220754118040203520552715563/1.3.6.1.4.1.9590.100.1.2.381440141511137044327302306604206077287/000000.dcm,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.207144238612220754118040203520552715563/1.3.6.1.4.1.9590.100.1.2.381440141511137044327302306604206077287/000001.dcm\n
2,P_00017,2,LEFT,CC,1,mass,ROUND,CIRCUMSCRIBED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.289610447411344525237308079592285912683/1.3.6.1.4.1.9590.100.1.2.22131189612893294827907969600765582967/000000.dcm,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.265130777712709757209861180301164730118/1.3.6.1.4.1.9590.100.1.2.212143028513012144941507232513982203672/000000.dcm,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.265130777712709757209861180301164730118/1.3.6.1.4.1.9590.100.1.2.212143028513012144941507232513982203672/000001.dcm\n
3,P_00017,2,LEFT,MLO,1,mass,ROUND,ILL_DEFINED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.22378064110931464632702821421529389998/1.3.6.1.4.1.9590.100.1.2.239949064412092068706566726490415129934/000000.dcm,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.66862495211464129620756742774012127297/1.3.6.1.4.1.9590.100.1.2.15403043813402510742192372832381918984/000000.dcm,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.66862495211464129620756742774012127297/1.3.6.1.4.1.9590.100.1.2.15403043813402510742192372832381918984/000001.dcm\n
4,P_00032,3,RIGHT,CC,1,mass,ROUND,OBSCURED,0,BENIGN,2,Mass-Test_P_00032_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.304236983211992984124654432490357131069/1.3.6.1.4.1.9590.100.1.2.215081818713600536113960661873725083371/000000.dcm,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.362221700813915332616463215463274950690/1.3.6.1.4.1.9590.100.1.2.199593071810497070809647901570077988031/000000.dcm,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.362221700813915332616463215463274950690/1.3.6.1.4.1.9590.100.1.2.199593071810497070809647901570077988031/000001.dcm\n
5,P_00032,3,RIGHT,MLO,1,mass,ROUND,OBSCURED,0,BENIGN,2,Mass-Test_P_00032_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.383460844111861145315621774372145221963/1.3.6.1.4.1.9590.100.1.2.318799084911119262430780458250312419361/000000.dcm,Mass-Test_P_00032_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.247343732911042250406750122270776949655/1.3.6.1.4.1.9590.100.1.2.44610919611642954332266410812181604922/000000.dcm,Mass-Test_P_00032_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.247343732911042250406750122270776949655/1.3.6.1.4.1.9590.100.1.2.44610919611642954332266410812181604922/000001.dcm\n
6,P_00037,3,RIGHT,CC,1,mass,ROUND,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00037_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.103263415712637900541513993551878074559/1.3.6.1.4.1.9590.100.1.2.345140832810160378520078721331878282316/000000.dcm,Mass-Test_P_00037_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.51371400012281630410382188863446338107/1.3.6.1.4.1.9590.100.1.2.335564193512609498716387099372607181452/000000.dcm

In [3]:
print(df2.info())
print(df2['image file path'][75])
print(df2['cropped image file path'][3])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   patient_id               378 non-null    object
 1   breast_density           378 non-null    int64 
 2   left or right breast     378 non-null    object
 3   image view               378 non-null    object
 4   abnormality id           378 non-null    int64 
 5   abnormality type         378 non-null    object
 6   mass shape               378 non-null    object
 7   mass margins             361 non-null    object
 8   assessment               378 non-null    int64 
 9   pathology                378 non-null    object
 10  subtlety                 378 non-null    int64 
 11  image file path          378 non-null    object
 12  cropped image file path  378 non-null    object
 13  ROI mask file path       378 non-null    object
dtypes: int64(4), object(10)
memory usage: 41.5

In [4]:
df3 = pd.read_csv('meta.csv')
df3.head(20)

,SeriesInstanceUID,StudyInstanceUID,Modality,SeriesDescription,BodyPartExamined,SeriesNumber,Collection,Visibility,ImageCount
0,1.3.6.1.4.1.9590.100.1.2.117041576511324414842508325652101471266,1.3.6.1.4.1.9590.100.1.2.229361142710768138411679379233064924540,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
1,1.3.6.1.4.1.9590.100.1.2.43873839610761788013224723323225482381,1.3.6.1.4.1.9590.100.1.2.195593486612988388325770883972107282733,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
2,1.3.6.1.4.1.9590.100.1.2.76741674113167646338262765132488965294,1.3.6.1.4.1.9590.100.1.2.257901172612530623323924356380431605062,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
3,1.3.6.1.4.1.9590.100.1.2.296931352612305599800425693833437698626,1.3.6.1.4.1.9590.100.1.2.109468616710242115222536802734027827120,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
4,1.3.6.1.4.1.9590.100.1.2.43665767012035310007732414810147712942,1.3.6.1.4.1.9590.100.1.2.380627129513562450304304820723973964760,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
5,1.3.6.1.4.1.9590.100.1.2.245633900110007082034118990512969470333,1.3.6.1.4.1.9590.100.1.2.143790815011189157220543617630697462001,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
6,1.3.6.1.4.1.9590.100.1.2.45131627713144582424202640662752027993,1.3.6.1.4.1.9590.100.1.2.310228111811303452111342857131681135070,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2
7,1.3.6.1.4.1.9590.100.1.2.321062807811123845106490625560193258924,1.3.6.1.4.1.9590.100.1.2.195655760513031195523320296122940320213,MG,cropped images,BREAST,1,CBIS-DDSM,1,1
8,1.3.6.1.4.1.9590.100.1.2.203989029910964209440425982812598330086,1.3.6.1.4.1.9590.100.1.2.222512969612930058132185640213545114412,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,1
9,1.3.6.1.4.1.9590.100.1.2.59331645212722964919598145441897056623,1.3.6.1.4.1.9590.100.1.2.407538725111828839342756496710990319312,MG,ROI mask images,BREAST,1,CBIS-DDSM,1,2


In [5]:
print(df3.info())
print(df3['SeriesInstanceUID'][3])
print(df3['StudyInstanceUID'][3])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6775 entries, 0 to 6774
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SeriesInstanceUID  6775 non-null   object
 1   StudyInstanceUID   6775 non-null   object
 2   Modality           6775 non-null   object
 3   SeriesDescription  6775 non-null   object
 4   BodyPartExamined   6775 non-null   object
 5   SeriesNumber       6775 non-null   int64 
 6   Collection         6775 non-null   object
 7   Visibility         6775 non-null   int64 
 8   ImageCount         6775 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 476.5+ KB
None
1.3.6.1.4.1.9590.100.1.2.296931352612305599800425693833437698626
1.3.6.1.4.1.9590.100.1.2.109468616710242115222536802734027827120


In [6]:
df4 = pd.read_csv('dicom_info.csv')
df4.head(20)


,file_path,image_path,AccessionNumber,BitsAllocated,BitsStored,BodyPartExamined,Columns,ContentDate,ContentTime,ConversionType,HighBit,InstanceNumber,LargestImagePixelValue,Laterality,Modality,PatientBirthDate,PatientID,PatientName,PatientOrientation,PatientSex,PhotometricInterpretation,PixelRepresentation,ReferringPhysicianName,Rows,SOPClassUID,SOPInstanceUID,SamplesPerPixel,SecondaryCaptureDeviceManufacturer,SecondaryCaptureDeviceManufacturerModelName,SeriesDescription,SeriesInstanceUID,SeriesNumber,SmallestImagePixelValue,SpecificCharacterSet,StudyDate,StudyID,StudyInstanceUID,StudyTime
0,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.129308726812851964007517874181459556304/1-172.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.129308726812851964007517874181459556304/1-172.jpg,NaN,16,16,BREAST,351,20160426,131732.685,WSD,15,1,65535,R,MG,NaN,Mass-Training_P_01265_RIGHT_MLO_1,Mass-Training_P_01265_RIGHT_MLO_1,MLO,NaN,MONOCHROME2,0,NaN,289,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.426380754911844882201419900442081103076,1,MathWorks,MATLAB,cropped images,1.3.6.1.4.1.9590.100.1.2.129308726812851964007517874181459556304,1,23078,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.271867287611061855725036643043149877819,214951.0
1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213277814849/1-249.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213277814849/1-249.jpg,NaN,16,16,BREAST,3526,20160426,143829.101,WSD,15,1,65535,R,MG,NaN,Mass-Training_P_01754_RIGHT_CC,Mass-Training_P_01754_RIGHT_CC,CC,NaN,MONOCHROME2,0,NaN,6256,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.235715018911442345533339224491594398571,1,MathWorks,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213277814849,1,0,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.161516517311681906612443997862211969669,193426.0
2,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.267213171011171858918434139331210917771/1-032.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.267213171011171858918434139331210917771/1-032.jpg,NaN,16,16,BREAST,1546,20160503,111956.298,WSD,15,1,65535,R,MG,NaN,Calc-Training_P_00232_RIGHT_CC,Calc-Training_P_00232_RIGHT_CC,CC,NaN,MONOCHROME2,0,NaN,4126,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.212917405611495375422194124032450014293,1,MathWorks,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.267213171011171858918434139331210917771,1,0,ISO_IR 100,20160807.0,DDSM,1.3.6.1.4.1.9590.100.1.2.291043622711253836701017460903623585574,161814.0
3,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/1-052.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/1-052.jpg,NaN,16,16,BREAST,97,20160503,115347.770,WSD,15,1,65535,L,MG,NaN,Calc-Test_P_00562_LEFT_CC_2,Calc-Test_P_00562_LEFT_CC_2,CC,NaN,MONOCHROME2,0,NaN,97,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.405776661412249467913600188550817314766,1,MathWorks,MATLAB,cropped images,1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317,1,32298,ISO_IR 100,20170829.0,DDSM,1.3.6.1.4.1.9590.100.1.2.335006093711888937440665960211609580159,180109.0
4,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/2-204.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/2-204.jpg,NaN,8,8,Left Breast,3104,20160503,115347.770,WSD,7,1,255,NaN,MG,NaN,P_00562_LEFT_CC_2.dcm,P_00562^P_00562,CC,NaN,MONOCHROME2,0,NaN,4560,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.418434643810489919922754459191670588257,1,MathWorks,MATLAB,NaN,1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317,1,0,ISO_IR 100,NaN,DDSM,1.3.6.1.4.1.9590.100.1.2.335006093711888937440665960211609580159,NaN
5,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.153339052913121382622526066491844156138/2-270.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.153339052913121382622526066491844156138/2-270.jpg,NaN,8,8,BREAST,1981,20160503,111620.055,WSD,7,1,255,R,MG,NaN,Calc-Tr

In [7]:
print(df4['file_path'][3])
print(df4['image_path'][3])

CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/1-052.dcm
CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187369611524586537789902641525311317/1-052.jpg


Podemos ver que hay un csv que contiene fotos con lesiones con calcificaciones, donde te dice el tipo de anormalidad, el tipo de calcificación, su distribución, otros datos, su patología (benigno o maligno) y las rutas a cada imagen: la que es mmografía completa, la que es recortada en la región de interés, y la máscara binaria del tumor píxel a píxel. También nos dicen el ID del paciente para cada foto, si es la teta izquierda o derecha y el punto de vista de la imagen (CC es desde arriba y MLO de perfil). Por tanto, para empezar conviene separar las imágenes que son mamografía completa de las que son recortes y de las máscaras. También convendría cambiarles el nombre a las imágenes (quizá poner info sobre el tumor en el propio nombre). 

OBSERVACIONES: El csv de calc y calc no parece ser por foto, sino por tipo de lesión, por eso hay tan pocos. En la image file path contiene una cosa rara, mezcla de varios IDs, no la ruta a una foto en concreto del dataset. De esa ruta hay que coger el studentUID y, con él, vamos al csv de dicom. En él cada fila sí corresponde a una única foto. Con ese studentUID identificamos todas las fotos con esa lesión, pues todas tienen asociado el mismo student ID. Hacemos un dataframe con todas las fotos y sus cosas. El que extrajo esto es subnormal profundo: es muy difícil identificar la lesión de cada imágenes con estos csv. Hay que hacer una combinación de varios para lograr algo.
ADEMÁS, la única informaciónn histológica fuerte es BENING / MALIGN. Es decir, no podemos clasificar el tipo de tumor. Solo si es benigno o maligno. Esto es una mierda, la verdad. EL INBreast creo que tampoco te permite hacer eso.

In [8]:
# ============================
# 2. CARGAR CSVs
# ============================

mass = pd.read_csv('mass_case_description_train_set.csv')
meta = pd.read_csv('meta.csv')
dicom = pd.read_csv('dicom_info.csv')

# ============================
# 3. EXTRAER StudyUID y SeriesUID DEL image_file_path
# ============================

mass["StudyInstanceUID"] = mass["image file path"].str.split("/").str[1]
mass["SeriesInstanceUID"] = mass["image file path"].str.split("/").str[2]

# ============================
# 4. MERGE MASS ↔ META (para saber cuántas imágenes tiene la serie)
# ============================

mass_meta = mass.merge(meta, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")

# ============================
# 5. MERGE MASS_META ↔ DICOM_INFO (para obtener rutas JPEG reales)
# ============================

mass_full = mass_meta.merge(dicom, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")
mass_full.head(10)

,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,StudyInstanceUID,SeriesInstanceUID,Modality_x,SeriesDescription_x,BodyPartExamined_x,SeriesNumber_x,Collection,Visibility,ImageCount,file_path,image_path,AccessionNumber,BitsAllocated,BitsStored,BodyPartExamined_y,Columns,ContentDate,ContentTime,ConversionType,HighBit,InstanceNumber,LargestImagePixelValue,Laterality,Modality_y,PatientBirthDate,PatientID,PatientName,PatientOrientation,PatientSex,PhotometricInterpretation,PixelRepresentation,ReferringPhysicianName,Rows,SOPClassUID,SOPInstanceUID,SamplesPerPixel,SecondaryCaptureDeviceManufacturer,SecondaryCaptureDeviceManufacturerModelName,SeriesDescription_y,SeriesNumber_y,SmallestImagePixelValue,SpecificCharacterSet,StudyDate,StudyID,StudyTime
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.422112722213189649807611434612228974994/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/000000.dcm,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.108268213011361124203859148071588939106/1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650/000000.dcm,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.108268213011361124203859148071588939106/1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650/000001.dcm\n,1.3.6.1.4.1.9590.100.1.2.422112722213189649807611434612228974994,1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515,MG,full mammogram images,BREAST,1,CBIS-DDSM,1,1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/1-211.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/1-211.jpg,NaN,16,16,BREAST,3024,20160414,121658.138,WSD,15,1,65535,L,MG,NaN,Mass-Training_P_00001_LEFT_CC,Mass-Training_P_00001_LEFT_CC,CC,NaN,MONOCHROME2,0,NaN,4808,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.156556873010981646517128874312129349516,1,MathWorks,MATLAB,full mammogram images,1,0,ISO_IR 100,20160720.0,DDSM,181327.0
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.319478999311971442426185353560182990988/1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834/000000.dcm,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.188473874511440575807446266233629582526/1.3.6.1.4.1.9590.100.1.2.227955274711225756835838775062793186053/000000.dcm,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.188473874511440575807446266233629582526/1.3.6.1.4.1.9590.100.1.2.227955274711225756835838775062793186053/000001.dcm\n,1.3.6.1.4.1.9590.100.1.2.319478999311971442426185353560182990988,1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834,MG,full mammogram images,BREAST,1,CBIS-DDSM,1,1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834/1-207.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834/1-207.jpg,NaN,16,16,BREAST,2656,20160414,121707.636,WSD,15,1,65535,L,MG,NaN,Mass-Training_P_00001_LEFT_MLO,Mass-Training_P_00001_LEFT_MLO,MLO,NaN,MONOCHROME2,0,NaN,4800,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.16927317613478383512517710492717097139,1,MathWorks,MATLAB,full mammogram images,1,0,ISO_IR 100,20160720.0,DDSM,181329.0
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.347107867812656628709864319310977895697/1.3.6.1.4.1.9590.100.1.2.89180046211022531834352631483669346540/000000.dcm,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.414182170112396175925115449620455230167/1.3.6.1.4.1.9590.100.1.2.429120414011832984817094399141838850375/000000.dcm,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.414182170112396175925115449

In [9]:
mass_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1318 entries, 0 to 1317
Data columns (total 59 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   patient_id                                   1318 non-null   object 
 1   breast_density                               1318 non-null   int64  
 2   left or right breast                         1318 non-null   object 
 3   image view                                   1318 non-null   object 
 4   abnormality id                               1318 non-null   int64  
 5   abnormality type                             1318 non-null   object 
 6   mass shape                                   1314 non-null   object 
 7   mass margins                                 1275 non-null   object 
 8   assessment                                   1318 non-null   int64  
 9   pathology                                    1318 non-null   object 
 10  

In [10]:
#Comprobamos que la info tiene sentido. Nos quedamos con las columnas importantes:
columnas_importantes = ['patient_id', 'PatientID', 'left or right breast', 
                        'breast_density', 'abnormality type', 'image file path', 'image_path',
                        'image view','SeriesDescription_x','SeriesDescription_y']
mass_full[columnas_importantes].head(7)

,patient_id,PatientID,left or right breast,breast_density,abnormality type,image file path,image_path,image view,SeriesDescription_x,SeriesDescription_y
0,P_00001,Mass-Training_P_00001_LEFT_CC,LEFT,3,mass,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.422112722213189649807611434612228974994/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/1-211.jpg,CC,full mammogram images,full mammogram images
1,P_00001,Mass-Training_P_00001_LEFT_MLO,LEFT,3,mass,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.319478999311971442426185353560182990988/1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834/1-207.jpg,MLO,full mammogram images,full mammogram images
2,P_00004,Mass-Training_P_00004_LEFT_CC,LEFT,3,mass,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.347107867812656628709864319310977895697/1.3.6.1.4.1.9590.100.1.2.89180046211022531834352631483669346540/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.89180046211022531834352631483669346540/1-250.jpg,CC,full mammogram images,full mammogram images
3,P_00004,Mass-Training_P_00004_LEFT_MLO,LEFT,3,mass,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.272600286511817402806912403581910920939/1.3.6.1.4.1.9590.100.1.2.295360926313492745441868049270168300162/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.295360926313492745441868049270168300162/1-067.jpg,MLO,full mammogram images,full mammogram images
4,P_00004,Mass-Training_P_00004_RIGHT_MLO,RIGHT,3,mass,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.174047308712169195014610267031196524486/1.3.6.1.4.1.9590.100.1.2.410524754913057908920631336070876889890/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.410524754913057908920631336070876889890/1-056.jpg,MLO,full mammogram images,full mammogram images
5,P_00009,Mass-Training_P_00009_RIGHT_CC,RIGHT,3,mass,Mass-Training_P_00009_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.348094436212980762312744999743818171955/1.3.6.1.4.1.9590.100.1.2.392091931911637760938815694332198115839/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.392091931911637760938815694332198115839/1-275.jpg,CC,full mammogram images,full mammogram images
6,P_00009,Mass-Training_P_00009_RIGHT_MLO,RIGHT,3,mass,Mass-Training_P_00009_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.255880668112335952601613726063195485525/1.3.6.1.4.1.9590.100.1.2.18553022011298363903753970133853455201/000000.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.18553022011298363903753970133853455201/1-104.jpg,MLO,full mammogram images,full mammogram images


In [11]:
print(mass_full['SeriesDescription_x'].value_counts())
print(mass_full['SeriesDescription_y'].value_counts())

SeriesDescription_x
full mammogram images    1318
Name: count, dtype: int64
SeriesDescription_y
full mammogram images    1318
Name: count, dtype: int64


In [24]:
#Vamos a cambiar el nombre de las imágenes que están en calc_full para que contengan toda la info que queremos
import shutil
import os

base_path = r'C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg'

def construir_ruta_real(image_path_csv):
    # 1. Separar por "/"
    partes = image_path_csv.split("/")

    # 2. El StudyInstanceUID está en la segunda parte
    study_uid = partes[2]

    # 3. El nombre del archivo está en la última parte, pero en Kaggle es .jpg
    filename = partes[-1]

    # 4. Construir ruta real
    ruta_real = os.path.join(base_path, study_uid, filename)

    return ruta_real
#También vamos a guardar solo el último filename

def guardar_filename(image_path_csv):
    # 1. Separar por "/"
    partes = image_path_csv.split("/")

    # 2. El StudyInstanceUID está en la segunda parte
    study_uid = partes[2]

    # 3. El nombre del archivo está en la última parte, pero en Kaggle es .jpg
    filename = partes[-1]

    # 4. Construir ruta real
    ruta_real = filename

    return ruta_real

In [25]:
mass_full["ruta_real"] = mass_full["image_path"].apply(construir_ruta_real)
mass_full['filename'] = mass_full['image_path'].apply(guardar_filename)

In [26]:
mass_full[['ruta_real','filename']].head(8)

,ruta_real,filename
0,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515\1-211.jpg,1-211.jpg
1,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.359308329312397897125630708681441180834\1-207.jpg,1-207.jpg
2,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.89180046211022531834352631483669346540\1-250.jpg,1-250.jpg
3,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.295360926313492745441868049270168300162\1-067.jpg,1-067.jpg
4,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.410524754913057908920631336070876889890\1-056.jpg,1-056.jpg
5,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.392091931911637760938815694332198115839\1-275.jpg,1-275.jpg
6,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.18553022011298363903753970133853455201\1-104.jpg,1-104.jpg
7,C:\Users\pedro\.cache\kagglehub\datasets\awsaf49\cbis-ddsm-breast-cancer-image-dataset\versions\1\jpeg\1.3.6.1.4.1.9590.100.1.2.399831242111800621220027542190666363688\1-046.jpg,1-046.jpg


In [27]:

    #Creamos el nombre que queremos hacer a la imagen
mass_full["nombre_imagenes"] = (
    mass_full["pathology"].astype(str) + "_" +
    mass_full["left or right breast"].astype(str) + "_" +
    mass_full["image view"].astype(str)  + "_" +
    mass_full['mass shape'].astype(str) + "_" +
    mass_full['mass margins'].astype(str) + "_" +
    mass_full['filename'].astype(str)
)
mass_full['nombre_imagenes'].head(8)


0     MALIGNANT_LEFT_CC_IRREGULAR-ARCHITECTURAL_DISTORTION_SPICULATED_1-211.jpg
1    MALIGNANT_LEFT_MLO_IRREGULAR-ARCHITECTURAL_DISTORTION_SPICULATED_1-207.jpg
2                 BENIGN_LEFT_CC_ARCHITECTURAL_DISTORTION_ILL_DEFINED_1-250.jpg
3                BENIGN_LEFT_MLO_ARCHITECTURAL_DISTORTION_ILL_DEFINED_1-067.jpg
4                                 BENIGN_RIGHT_MLO_OVAL_CIRCUMSCRIBED_1-056.jpg
5                                 MALIGNANT_RIGHT_CC_OVAL_ILL_DEFINED_1-275.jpg
6                                MALIGNANT_RIGHT_MLO_OVAL_ILL_DEFINED_1-104.jpg
7                 MALIGNANT_LEFT_MLO_IRREGULAR_ILL_DEFINED-SPICULATED_1-046.jpg
Name: nombre_imagenes, dtype: object

In [30]:
#Aplicamos con shutil
path_destino = r'C:\Users\pedro\Documents\MÁSTER UNIR 💻\TFM\extracción de imágenes del dataset\Imagenes'

for i in range(0,len(mass_full)):
    ruta_origen = mass_full['ruta_real'][i]
    ruta_destino_completa = os.path.join(path_destino, mass_full['nombre_imagenes'][i])
    shutil.copy(ruta_origen,ruta_destino_completa)

1. Esquema de combinación razonable
Opción fuerte y limpia:

INbreast → Segmentación supervisada

Entrenas una U‑Net/nnU‑Net para segmentar masas/calcificaciones.

Métricas: Dice, IoU, etc.

Dataset pequeño pero con GT perfecto.

INbreast → Clasificación a partir de máscaras o crops

A partir de las máscaras, generas:

crops alrededor de la lesión

o features agregadas (área, forma, etc.)

Entrenas un clasificador benigno/maligno (si las etiquetas lo permiten) o por tipo de lesión.

Kaggle CBIS → Clasificación en full mammograms

Usas solo las full mammo que ya tienes mapeadas.

Tarea: benigno/maligno, calc/calc, etc.

Modelo tipo ResNet/ViT.

Localización débil: Grad‑CAM sobre las full mammo.

Generalización cruzada

Entrenas en INbreast, pruebas en Kaggle (si las etiquetas son compatibles).

Entrenas en Kaggle, pruebas en INbreast.

Discutes dominio, cambio de distribución, diferencias de adquisición, etc.

In [31]:
#Hacemos lo mismo con calc
# ============================
# 2. CARGAR CSVs
# ============================

calc = pd.read_csv('calc_case_description_train_set.csv')
meta = pd.read_csv('meta.csv')
dicom = pd.read_csv('dicom_info.csv')

# ============================
# 3. EXTRAER StudyUID y SeriesUID DEL image_file_path
# ============================

calc["StudyInstanceUID"] = calc["image file path"].str.split("/").str[1]
calc["SeriesInstanceUID"] = calc["image file path"].str.split("/").str[2]

# ============================
# 4. MERGE calc ↔ META (para saber cuántas imágenes tiene la serie)
# ============================

calc_meta = calc.merge(meta, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")

# ============================
# 5. MERGE calc_META ↔ DICOM_INFO (para obtener rutas JPEG reales)
# ============================

calc_full = calc_meta.merge(dicom, on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")
calc_full.head(10)

,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path,StudyInstanceUID,SeriesInstanceUID,Modality_x,SeriesDescription_x,BodyPartExamined_x,SeriesNumber_x,Collection,Visibility,ImageCount,file_path,image_path,AccessionNumber,BitsAllocated,BitsStored,BodyPartExamined_y,Columns,ContentDate,ContentTime,ConversionType,HighBit,InstanceNumber,LargestImagePixelValue,Laterality,Modality_y,PatientBirthDate,PatientID,PatientName,PatientOrientation,PatientSex,PhotometricInterpretation,PixelRepresentation,ReferringPhysicianName,Rows,SOPClassUID,SOPInstanceUID,SamplesPerPixel,SecondaryCaptureDeviceManufacturer,SecondaryCaptureDeviceManufacturerModelName,SeriesDescription_y,SeriesNumber_y,SmallestImagePixelValue,SpecificCharacterSet,StudyDate,StudyID,StudyTime
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157/1.3.6.1.4.1.9590.100.1.2.47414316010368386519740343172775938548/000000.dcm,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.328778919012412769218080124214088709081/1.3.6.1.4.1.9590.100.1.2.393344010211719049419601138200355094682/000001.dcm\n,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9590.100.1.2.328778919012412769218080124214088709081/1.3.6.1.4.1.9590.100.1.2.393344010211719049419601138200355094682/000000.dcm,1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157,1.3.6.1.4.1.9590.100.1.2.47414316010368386519740343172775938548,MG,full mammogram images,BREAST,1,CBIS-DDSM,1,1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.47414316010368386519740343172775938548/1-188.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.47414316010368386519740343172775938548/1-188.jpg,NaN,16,16,BREAST,2761,20160503,105435.478,WSD,15,1,65535,R,MG,NaN,Calc-Training_P_00005_RIGHT_CC,Calc-Training_P_00005_RIGHT_CC,CC,NaN,MONOCHROME2,0,NaN,5056,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.378776769711002686209961163603436313519,1,MathWorks,MATLAB,full mammogram images,1,0,ISO_IR 100,20160807.0,DDSM,161037.0
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.9590.100.1.2.427517897311902339923511678883689433338/1.3.6.1.4.1.9590.100.1.2.250596608311207922527805669693579696727/000000.dcm,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.67512362210319636108148504382680781938/1.3.6.1.4.1.9590.100.1.2.296281207812130400303493285473798422894/000001.dcm\n,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1.9590.100.1.2.67512362210319636108148504382680781938/1.3.6.1.4.1.9590.100.1.2.296281207812130400303493285473798422894/000000.dcm,1.3.6.1.4.1.9590.100.1.2.427517897311902339923511678883689433338,1.3.6.1.4.1.9590.100.1.2.250596608311207922527805669693579696727,MG,full mammogram images,BREAST,1,CBIS-DDSM,1,1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.250596608311207922527805669693579696727/1-189.dcm,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.250596608311207922527805669693579696727/1-189.jpg,NaN,16,16,BREAST,2836,20160503,105440.625,WSD,15,1,65535,R,MG,NaN,Calc-Training_P_00005_RIGHT_MLO,Calc-Training_P_00005_RIGHT_MLO,MLO,NaN,MONOCHROME2,0,NaN,5386,1.2.840.10008.5.1.4.1.1.7,1.3.6.1.4.1.9590.100.1.2.344560194211255710311153402222519334809,1,MathWorks,MATLAB,full mammogram images,1,0,ISO_IR 100,20160807.0,DDSM,161039.0
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.201322325113694962619881476352450072222/1.3.6.1.4.1.9590.100.1.2.228699627313487111012474405462022068297/000000.dcm,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880/1.3.6.1.4.1.9590.100.1.2.314135871111943890422150247820137952041/000001.dcm\n,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880/1.3.6.1.4.1.9590

In [34]:
calc_full["ruta_real"] = calc_full["image_path"].apply(construir_ruta_real)
calc_full['filename'] = calc_full['image_path'].apply(guardar_filename)

    #Creamos el nombre que queremos hacer a la imagen
calc_full["nombre_imagenes"] = (
    calc_full["pathology"].astype(str) + "_" +
    calc_full["left or right breast"].astype(str) + "_" +
    calc_full["image view"].astype(str)  + "_" +
    calc_full['calc type'].astype(str) + "_" +
    calc_full['calc distribution'].astype(str) + "_" +
    calc_full['filename'].astype(str)
)
calc_full['nombre_imagenes'].head(8)

0           MALIGNANT_RIGHT_CC_AMORPHOUS_CLUSTERED_1-188.jpg
1          MALIGNANT_RIGHT_MLO_AMORPHOUS_CLUSTERED_1-189.jpg
2                BENIGN_LEFT_CC_PLEOMORPHIC_LINEAR_1-190.jpg
3               BENIGN_LEFT_MLO_PLEOMORPHIC_LINEAR_1-191.jpg
4     BENIGN_WITHOUT_CALLBACK_LEFT_CC_nan_REGIONAL_1-192.jpg
5     BENIGN_WITHOUT_CALLBACK_LEFT_CC_nan_REGIONAL_1-192.jpg
6     BENIGN_WITHOUT_CALLBACK_LEFT_CC_nan_REGIONAL_1-192.jpg
7    BENIGN_WITHOUT_CALLBACK_LEFT_MLO_nan_REGIONAL_1-193.jpg
Name: nombre_imagenes, dtype: object

In [35]:
#Aplicamos con shutil
path_destino = r'C:\Users\pedro\Documents\MÁSTER UNIR 💻\TFM\extracción de imágenes del dataset\Imagenes Calcification'

for i in range(0,len(calc_full)):
    ruta_origen = calc_full['ruta_real'][i]
    ruta_destino_completa = os.path.join(path_destino, calc_full['nombre_imagenes'][i])
    shutil.copy(ruta_origen,ruta_destino_completa)